# 01 — Carico ed esploro VIX / MOVE

Dati GRATUITI (vedi `fetch_data.py`, esegui prima `python fetch_data.py`):
VIX (CBOE, dal 1990), MOVE (Yahoo), benchmark SPY/TLT/GLD/DBC.
I dati sono in `data_local/` e **non** sono versionati.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import vol_regime as vr
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

In [2]:
vol = vr.load_vol(); prices = vr.load_prices()
print("VIX/MOVE:", vol.dropna().index.min().date(), "→", vol.index.max().date())
vol.describe(percentiles=[.5,.7,.8,.9,.95]).T

VIX/MOVE: 2002-11-12 → 2026-06-26


,count,mean,std,min,50%,70%,80%,90%,95%,max
VIX,9216.000,19.448,7.738,9.140,17.610,21.510,24.190,28.555,32.942,82.690
MOVE,5840.000,86.997,30.568,36.620,80.300,99.703,111.384,126.505,142.225,264.600


## Percentili (le soglie candidate vengono da qui)

In [3]:
pd.DataFrame({c: vol[c].dropna().quantile([.5,.7,.8,.9,.95]) for c in vol.columns})

,VIX,MOVE
0.500,17.610,80.300
0.700,21.510,99.703
0.800,24.190,111.384
0.900,28.555,126.505
0.950,32.942,142.225


## Relazione vol ↔ drawdown di mercato (la vol esplode nei crash)

In [4]:
spy = prices["SPY"].dropna()
dd = spy/spy.cummax() - 1.0
joined = pd.concat({"VIX": vol["VIX"], "SPY_drawdown": dd}, axis=1).dropna()
print("corr(VIX, drawdown SPY):", round(joined["VIX"].corr(joined["SPY_drawdown"]), 3))
print("VIX medio quando drawdown < -20%:", round(joined.loc[joined.SPY_drawdown < -0.2, "VIX"].mean(), 1))
print("VIX medio quando drawdown >  -5%:", round(joined.loc[joined.SPY_drawdown > -0.05, "VIX"].mean(), 1))

corr(VIX, drawdown SPY): -0.576
VIX medio quando drawdown < -20%: 25.3
VIX medio quando drawdown >  -5%: 15.8


**Nota.** La VIX è fortemente correlata (negativamente) al drawdown: esplode
*durante* i crash. Questo è il punto chiave per i segnali sotto — la VIX è
**coincidente**, non necessariamente anticipatrice.